last modified date : 2026.07   
제작 : 모두의연구소 퍼실팀

# Day 2 실습 — Advanced·Modular RAG + RAGAS 평가

# 들어가며

Day 1에서는 가장 기본형인 **Naive RAG** 파이프라인을 직접 구현해 보았습니다. 이번 실습에서는 한국어 QA 벤치마크 **KorQuAD v1** 데이터셋 위에서 **Advanced·Modular RAG** 의 핵심 기법(Multi-Query, RAG-Fusion, HyDE, Reranking, Self-RAG)을 단계적으로 적용하고, 그 결과를 **RAGAS** 로 정량 평가합니다.

이번 실습이 끝나면 다음을 직접 말할 수 있게 됩니다.
- Naive RAG 대비 **어떤 단계**를 보강하면 정답률이 올라가는가
- Multi-Query / RAG-Fusion / HyDE / Reranker / Self-RAG 는 각각 **어떤 코드 라인**으로 적용하는가
- RAGAS 의 4대 지표(Faithfulness · Answer Relevance · Context Precision · Context Recall)는 어떻게 계산되고 어떻게 읽는가
- 내 RAG 가 ‘얼마나 좋아졌는지’를 **숫자로** 보여주는 방법

## Step 0 : 설치와 준비  
Day 1과 동일하게 Colab에서 진행한다고 가정합니다.

In [ ]:
# Colab pre-installed langchain 0.3 / ragas 0.1~0.4 를 ragas 0.2.10 호환 조합으로 정리합니다.
# 처음 실행 시 약 3~5분 걸립니다. 진행률 출력을 보면서 기다리세요 (멈춘 게 아닙니다).

# 1) 기존 langchain / ragas 패키지 제거 — 버전 충돌로 인한 pip resolver 백트래킹 방지
!pip uninstall -y ragas ragas-experimental langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-chroma

# 2) 0.2 시리즈 패치 버전까지 핀 설치 — resolver 부담 최소화 (-q 제거해서 진행률 보이게)
!pip install --no-cache-dir \
    "ragas==0.2.10" \
    "langchain==0.2.17" \
    "langchain-core==0.2.43" \
    "langchain-community==0.2.19" \
    "langchain-openai==0.1.25" \
    "langchain-text-splitters==0.2.4" \
    "langchain-chroma==0.1.4" \
    pypdf chromadb tiktoken sentence-transformers datasets nest_asyncio pandas

Found existing installation: langchain 1.3.13
Uninstalling langchain-1.3.13:
  Successfully uninstalled langchain-1.3.13
Found existing installation: langchain-core 1.4.9
Uninstalling langchain-core-1.4.9:
  Successfully uninstalled langchain-core-1.4.9
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 207.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 308.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

> ⚠️ **위 설치 셀(Step 0)을 실행한 뒤 반드시 [런타임 > 세션 다시 시작 (Restart session)]을 한 번 눌러주세요.**
>
> 이 셀은 Colab에 기본 설치된 langchain을 제거하고 `0.2.x` / `ragas 0.2.10` 조합으로 다운그레이드합니다. 이미 메모리에 로드된 패키지를 교체하는 것이라 Colab이 재시작을 요구합니다.
>
> 재시작 후에는 **설치 셀은 다시 실행하지 말고** 이 셀 아래(키 설정)부터 순서대로 실행하면 됩니다.

In [ ]:
import os
# chromadb 익명 통계 전송 끄기 — posthog SDK 인자 충돌로 ERROR 로그가 뜨는 것 방지
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import nest_asyncio
nest_asyncio.apply()  # RAGAS가 Colab의 비동기 이벤트 루프와 충돌하지 않도록

In [ ]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata

# 등록해둔 보안 비밀 이름이 뭐든 잡히게. 하나만 성공하면 됨
CANDIDATES = ["OPENAI_API_KEY", "OPENAI_KEY", "OPENAI_APIKEY", "openai_api_key"]
key = None
for name in CANDIDATES:
    try:
        key = userdata.get(name)
        print("보안 비밀 사용:", name)
        break
    except Exception as e:
        print(f"  {name}: {type(e).__name__}")

if not key:
    raise RuntimeError(
        "키를 못 찾았다. 왼쪽 열쇠 아이콘에서 (1) 이름 확인, (2) '노트북 액세스' 토글 ON 확인."
    )

os.environ["OPENAI_API_KEY"] = key
print("키 길이:", len(key), "| 접두사:", key[:7])   # sk-proj- 또는 sk- 로 시작해야 정상

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("GPU 런타임이 아니다. Step 7 reranker 와 Step 10 수집이 많이 느려진다.")

보안 비밀 사용: OPENAI_API_KEY
키 길이: 164 | 접두사: sk-proj
device: cuda


## Step 1 : KorQuAD v1 위에서 Naive RAG 베이스라인 만들기

Day 1에서 만든 RAG 파이프라인을 한국어 QA 벤치마크 **KorQuAD v1** 위에 다시 한 번 올립니다. 이후 단계는 모두 이 베이스라인 위에 ‘덧붙이는’ 방식입니다.

- HuggingFace `datasets` 로 KorQuAD v1 자동 다운로드 (별도 PDF 업로드 불필요)
- 일부만 샘플링해 토큰 비용 통제 (unique context 약 200개)
- Embedding → VectorStore → Retriever → LLM
- 검색 전략은 단순 `similarity` (top-k)

**📥 데이터셋**: <https://huggingface.co/datasets/KorQuAD/squad_kor_v1>

In [ ]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken, random

tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    return len(tokenizer.encode(text))

# 1) 데이터셋 로드 + 2000개 샘플링 + context 중복 제거 → unique 약 800개
#    (Vector DB 가 크면 Reranker 의 정밀도 개선 효과가 더 또렷하게 보입니다.
#     인덱싱 토큰 비용 약 0.01 USD 추가)
raw_ds = load_dataset("squad_kor_v1", split="validation").shuffle(seed=42).select(range(2000))

unique = {}
for ex in raw_ds:
    if ex["context"] not in unique:
        unique[ex["context"]] = ex["title"]
context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique.items()]

# 2) chunk 단위로 분할 (KorQuAD context는 짧지만 길이 균질화를 위해 splitter 사용)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)
docs = splitter.split_documents(context_docs)

# 3) Embedding & Chroma 적재 — chunk 약 800개를 한 번에 넣으면 chromadb 의 batch limit
#    (Colab 환경에서 보통 5461) 또는 OpenAI rate limit 에 걸릴 수 있어
#    100개씩 배치로 add_documents 합니다.
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(docs), BATCH):
    db.add_documents(docs[i:i+BATCH])

# 4) Retriever (Naive: similarity)
naive_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 5) LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"베이스라인 준비 완료 — unique context: {len(context_docs)}, chunks: {len(docs)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


베이스라인 준비 완료 — unique context: 847, chunks: 1264


베이스라인 RAG로 간단한 질의를 던져 답이 나오는지 확인합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "다음 문서를 참고해 질문에 한국어로 간결하게 답하세요. 문서에 없는 내용은 만들지 마세요.\n\n"
    "[문서]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 데이터셋에서 첫 질문 하나를 뽑아 테스트
TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("A:", naive_chain.invoke(TEST_Q))

Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


A: 대중교통체계입니다.


## Step 2 : Pre-retrieval 강화 — Multi-Query Retrieval  

사용자가 던진 질문 하나로만 검색하면 ‘다른 표현’으로 적힌 정답을 놓칠 수 있습니다. **Multi-Query Retrieval**은 LLM에게 ‘같은 의도의 다른 질문 N개’를 만들게 시킨 뒤, 각 질문으로 병렬 검색하고 결과를 합칩니다.

LangChain은 이를 한 클래스로 제공합니다.

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# 어떤 ‘유사 질문’으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['2004년 이명박 서울시장이 재직할 때 어떤 주요 개선 사항이 있었나요?  ', '이명박이 2004년에 서울시장으로서 추진한 주요 정책이나 변화는 무엇인가요?  ', '2004년 이명박 서울시장 재임 중에 이루어진 주요 개선 프로젝트는 어떤 것들이 있나요?']


검색된 문서 수: 6
---
2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재선에 도전했다. 6월 2일에 치뤄진 지방선거에서 개표 초반에 한명숙 후보에게 뒤지다가, 후반 강남 3구의 개표가 시작되면서 역전하여 민선 5기 제34대 서울특별시장으로 재선되었다. 구체적으로 강남구(+59,206, +25.68%), 서초구(+43,820, +23.66%), 송파구(+23,814, +8.19%), 강동구(+11,097, +5.33%), 용산구(+8,579, +8.24%), 양천구(+1,078, +0.51%), 영


## Step 2.5 : RAG-Fusion — Multi-Query + RRF로 묶어내기

Day2_1 노트에서 “꼭 짚고 가라”고 했던 패턴 중 하나가 **RAG-Fusion** 입니다. Step 2의 Multi-Query는 ‘유사 질문 N개로 병렬 검색’ 까지만 했는데, **RAG-Fusion** 은 그 N개 검색 결과를 **Reciprocal Rank Fusion (RRF)** 라는 간단한 공식으로 합쳐 ‘여러 쿼리에서 공통으로 상위에 떴던 문서’ 를 최상단으로 끌어올립니다.

RRF 점수 공식:

$$
\text{score}(d) = \sum_{i=1}^{N} \frac{1}{k + \text{rank}_i(d)}
$$

- $\text{rank}_i(d)$ : i번째 쿼리의 결과에서 문서 $d$ 의 순위 (1부터)
- $k$ : 스무딩 상수 (관례적으로 60)

아래 셀에서는 (1) sub-query 생성, (2) 각 sub-query 로 검색, (3) **RRF 함수는 여러분이 직접 채우기**, (4) 결과 확인까지 한 번에 해봅니다.

In [ ]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query 가 내부적으로 하는 일을 명시적으로 노출 (한국어)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 4개의 한국어 검색 쿼리를 만드세요. "
    "오직 4개의 쿼리만 한 줄에 하나씩 출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]


# (2) RRF 함수
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    # results_per_query : 쿼리별 검색 결과(각각 순위 순) / k : 스무딩 상수 / top_k : 최종 반환 개수
    scores = defaultdict(float)
    docs_by_key = {}

    # 각 쿼리 결과를 순회하며 문서별 RRF 점수를 누적
    for docs in results_per_query:
        for rank, doc in enumerate(docs):        # enumerate 는 0부터라 아래에서 +1
            key = doc.page_content               # 같은 chunk 가 여러 쿼리에 등장하면 하나로 묶는 기준
            scores[key] += 1.0 / (k + rank + 1)
            docs_by_key[key] = doc

    # 누적 점수 내림차순으로 상위 top_k 반환
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [docs_by_key[key] for key, _ in ranked[:top_k]]


# (3) 한 번 돌려보기 — 원본 질문을 함께 넣는다.
#     LLM 이 만든 변형만 쓰면 원 질문의 초점에서 통째로 이탈해도 알아챌 방법이 없다.
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for q in sub_queries:
    print(" -", q)

all_queries = [TEST_Q] + sub_queries
results_per_q = [db.similarity_search(q, k=5) for q in all_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300] if fused else "(결과 없음)")

확장 질문 4개:
 - 2004년 이명박 서울시장 재직 중 개선한 사항은?
 - 이명박이 2004년 서울시장으로서 개선한 내용은 무엇인가?
 - 2004년 서울시장 이명박이 전면적으로 개선한 것은 어떤 것인가?
 - 이명박 서울시장 재직 시절 2004년에 개선한 것은 무엇인지?

RAG-Fusion top-1 문서:
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교


**왜 유사도 점수를 그냥 더하면 안 되는지**

처음에는 각 쿼리의 유사도 점수를 그대로 합산하면 되지 않나 생각했는데, 찾아보니 그러면 안 되는 이유가 있었습니다.
쿼리마다 유사도 스케일이 달라서, 어떤 쿼리는 최고점이 0.82인데 어떤 쿼리는 0.61입니다.
그대로 더하면 전반적으로 점수가 후하게 나온 쿼리가 결과를 독식하게 됩니다.
순위는 쿼리끼리 비교가 되는 공통 단위라, 서로 기준이 다른 랭커를 섞을 때는 rank 기반 융합을 쓴다고 합니다.
BM25와 벡터 검색을 섞는 하이브리드 검색에서도 같은 이유로 RRF를 쓴다고 하니 이해가 됐습니다.

상수 $k$는 1등과 2등의 점수 격차를 정하는 값인 것 같습니다.
$k$가 작으면 각 쿼리의 1등이 거의 다 먹고, 크면 순위차가 뭉개져서 몇 개 쿼리에 등장했는지가 더 중요해집니다.

RRF는 API 호출이 없는 순수 함수라서 정답을 아는 가짜 입력으로 먼저 검증해봤습니다.
실제 검색 결과로만 테스트하면 함수가 틀린 건지 검색이 이상한 건지 구분이 안 될 것 같았습니다.

In [ ]:
class _FakeDoc:
    def __init__(self, c): self.page_content = c

A, B, C, D = (_FakeDoc(x) for x in "ABCD")

# case1: 1등을 한 번 한 B 보다, 2등을 세 번 한 A 가 위로 올라와야 한다
r1 = reciprocal_rank_fusion([[B, A], [C, A], [D, A]], top_k=4)
print("case1:", [d.page_content for d in r1])
assert r1[0].page_content == "A"

# case2: 쿼리가 하나면 원래 순위가 그대로 보존돼야 한다 (융합이 순위를 망가뜨리지 않는지)
r2 = reciprocal_rank_fusion([[C, A, B]], top_k=3)
print("case2:", [d.page_content for d in r2])
assert [d.page_content for d in r2] == ["C", "A", "B"]

# case3: 빈 입력에서 IndexError 로 죽지 않아야 한다
print("case3:", reciprocal_rank_fusion([], top_k=3), reciprocal_rank_fusion([[]], top_k=3))
print("RRF 검증 통과")

case1: ['A', 'B', 'C', 'D']
case2: ['C', 'A', 'B']
case3: [] []
RRF 검증 통과


## Step 3 : 패턴 ② HyDE — 가상의 ‘정답’으로 진짜 정답 찾기  

질문은 짧은 의문문, 정답은 긴 평서문이라 둘의 임베딩이 의외로 멀 수 있습니다. **HyDE(Hypothetical Document Embeddings)** 는 검색 전에 LLM에게 ‘가상의 정답’을 쓰게 한 뒤, 그 가상 답변을 임베딩해서 검색합니다.

직접 구현해 보겠습니다.

In [ ]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 해당 분야 전문가입니다. 다음 질문에 대해 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def hyde_retrieve(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 검색"""
    hypothetical = hyde_generator.invoke({"question": question})
    return db.similarity_search(hypothetical, k=k), hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 답변(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200])

가상 답변(HyDE):
 2004년 이명박이 서울시장으로 재직하던 시절, 그는 서울시의 교통 체계를 전면적으로 개선하는 데 주력했습니다. 특히, 그는 '서울시 교통체계 개선 종합계획'을 수립하여 대중교통의 효율성을 높이고, 도로 혼잡을 줄이기 위한 다양한 정책을 시행했습니다. 이 과정에서 지하철 노선 확장과 버스 전용차선 도입, 그리고 자전거 도로의 확충 등이 이루어졌습니다. 이러한 노력은 서울시민의 교통 편의성을 크게 향상시키고, 대기 오염 문제 해결에도 기여했습니다. 
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 4 : Post-retrieval 강화 — Cross-Encoder Reranking (multilingual)

검색 결과를 그대로 LLM 에 넘기지 않고, **Cross-encoder reranker** 가 (질문, 문단)을 함께 보면서 진짜 관련도를 다시 점수화합니다. 정밀도가 15~30% 개선되는 게 일반적인 보고입니다.

한국어 문서를 다루고 있으므로 다국어를 지원하는 cross-encoder 를 사용합니다. `BAAI/bge-reranker-v2-m3` 는 한국어를 포함한 100개 이상 언어에서 동작합니다. 처음 실행 시 모델 다운로드(~2GB)가 발생합니다.

In [ ]:
from sentence_transformers import CrossEncoder

# 다국어 cross-encoder (한국어 포함)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환"""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3[0].page_content[:200])

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 5 : Advanced RAG 체인 조립  

위에서 만든 컴포넌트들을 하나의 체인으로 묶습니다. **‘넓게 검색 → Reranker로 좁히기 → LLM 답변’** 패턴이 가장 흔히 쓰입니다.

In [ ]:
def advanced_rag(question):
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(question)
    # 2) Cross-encoder 로 진짜 관련도 재정렬 후 상위 3개
    top = rerank(question, candidates, top_k=3)
    # 3) 프롬프트에 컨텍스트로 주입 → 답변
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("Advanced RAG 답변:\n", ans_adv)

Advanced RAG 답변:
 대중교통체계입니다.


## Step 5.5 : Self-RAG — 검색 필요성 판단 + 답변 자가 비평

Day2_1 노트에서 강조한 또 하나의 핵심 패턴, **Self-RAG** 입니다. Self-RAG의 핵심은 **LLM이 검색·답변 과정에 스스로 비평(critique)을 끼워 넣는다**는 점입니다.

이번 셀에서는 공식 Self-RAG 모델을 따로 받지 않고, **세 개의 작은 LLM 프롬프트**로 같은 흐름을 흉내내 봅니다.

1. **Retrieve 결정** — 질문이 들어오면, 외부 검색이 필요한지 LLM이 먼저 판단합니다. (`YES`/`NO` 한 단어)
2. **답변 생성** — `YES` 면 일반 RAG, `NO` 면 검색 없이 LLM 단독 답변.
3. **답변 자가 비평** — 생성된 답변이 컨텍스트에 충분히 근거하는지 LLM이 점검합니다. (`SUPPORTED` / `NOT_SUPPORTED`)
4. **보완 재시도** — `NOT_SUPPORTED` 면 Step 3의 **HyDE** 로 검색 쿼리를 바꿔 한 번 더 시도합니다.

코드 골격은 제공해 두었고, **두 군데 핵심 프롬프트만 여러분이 직접 채워주세요.**

In [ ]:
# Self-RAG : retrieve 판단 + 자가 비평 + HyDE 재시도

# (1) 검색 필요성 판단 프롬프트
#     판정은 반드시 한 단어로 강제한다. 문장으로 답하게 두면 아래 startswith("NO") / "NOT" in ... 파싱이
#     "이 답변은 문서에 없는 내용을 포함하지 않습니다" 같은 문장에서 오작동한다.
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    "당신은 질문을 분류하는 라우터입니다. 아래 질문에 답하기 위해 외부 문서 검색이 필요한지 판단하세요.\n"
    "- 특정 인물, 사건, 기관, 연도, 수치처럼 문서에서 확인해야 하는 사실이 필요하면 YES\n"
    "- 일반 상식, 단순 계산, 용어의 뜻처럼 당신이 이미 아는 지식만으로 답할 수 있으면 NO\n"
    "설명이나 문장을 쓰지 말고 YES 또는 NO 한 단어만 출력하세요.\n\n"
    "질문: {question}\n판단:"
)

# (2) 답변 자가 비평 프롬프트
#     "문장이 짧다/불완전하다" 를 근거 부족으로 오해하지 않도록 판정 기준을 명시적으로 좁힌다.
#     KorQuAD 정답은 '대중교통체계' 같은 명사구라 이 단서가 없으면 정상 답변도 반려된다.
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    "[답변]이 [문서]에 근거하는지 판정하세요.\n"
    "- 답변이 명사구나 짧은 구절이어도, 그 내용이 문서에서 확인되면 SUPPORTED\n"
    "- 문장이 완결되지 않았다거나 설명이 부족하다는 이유로 NOT_SUPPORTED 를 주지 마세요\n"
    "- 문서에 없는 사실이 들어갔거나 문서와 어긋날 때만 NOT_SUPPORTED\n"
    "설명 없이 SUPPORTED 또는 NOT_SUPPORTED 한 단어만 출력하세요.\n\n"
    "[문서]\n{context}\n\n[답변]\n{answer}\n\n판정:"
)


def self_rag(question, max_retries=1, verbose=True):
    decision = (RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}).strip().upper()
    if verbose:
        print(f"[1] Retrieve 필요? -> {decision}")

    if decision.startswith("NO"):
        ans = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변 사용")
        return ans, []

    docs = db.as_retriever(search_kwargs={"k": 3}).invoke(question)

    for attempt in range(max_retries + 1):
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "question": question})
        critique = (CRITIQUE_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "answer": answer}).strip().upper()
        if verbose:
            print(f"[3] 시도 {attempt+1} — 자가 비평: {critique}")

        if "NOT" not in critique:
            return answer, docs

        if attempt < max_retries:
            hyp = hyde_generator.invoke({"question": question})
            docs = db.similarity_search(hyp, k=3)
            if verbose:
                print("[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색")

    return answer, docs


ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)

[1] Retrieve 필요? -> YES
[3] 시도 1 — 자가 비평: SUPPORTED

=== Self-RAG 최종 답변 ===
대중교통체계입니다.


#### 프롬프트를 채우기 전에 한번 돌려봤는데

TODO를 비운 채로(프롬프트가 빈 문자열인 상태) 실행하면 에러가 나겠거니 했는데, 이런 로그가 나왔습니다.

```
[1] Retrieve 필요? -> HELLO! HOW CAN I ASSIST YOU TODAY?
[3] 시도 1 — 자가 비평: HELLO! HOW CAN I ASSIST YOU TODAY?

=== Self-RAG 최종 답변 ===
대중교통체계입니다.
```

빈 프롬프트를 받은 LLM이 인사를 했고, 그 인사말이 판정 결과 자리에 그대로 들어왔습니다.
그런데도 파이프라인은 끝까지 갔고 답까지 맞았습니다. 왜 그런지 따라가 보니 이랬습니다.

- `"HELLO!..."`는 `"NO"`로 시작하지 않아서 → 검색 분기로 들어감
- `"HELLO!..."` 안에 `"NOT"`이 없어서 → SUPPORTED로 처리됨
- 그래서 최종 답변은 정상 출력

예외도 안 나고 답도 맞는데 Self-RAG는 전혀 동작하지 않은 상태였습니다.
판정을 문자열 포함 여부로 결정하면 판정기가 망가져도 이렇게 조용히 통과할 수 있다는 걸 알게 됐습니다.
출력만 봐서는 알 수 없으니, 정답을 아는 통제 입력으로 분기를 직접 찔러보는 셀을 아래에 추가했습니다.

In [ ]:
# 라우터가 두 갈래를 실제로 다 타는지 확인.
# 프롬프트가 애매하면 모델이 전부 YES 를 뱉고 NO 분기는 한 번도 실행되지 않는 죽은 코드가 되는데,
# 그래도 파이프라인은 멀쩡히 돌아가고 로그도 정상이라 눈치채기 어렵다.
probe = [
    ("2 더하기 3은 얼마인가?", "NO"),
    ("파이썬에서 리스트와 튜플의 차이는?", "NO"),
    (raw_ds[0]["question"], "YES"),
    (raw_ds[1]["question"], "YES"),
]

router = RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()
for q, expect in probe:
    got = router.invoke({"question": q}).strip().upper()
    mark = "ok" if got.startswith(expect) else "MISMATCH"
    print(f"[{mark:8s}] 기대={expect:3s} 실제={got:3s} | {q[:40]}")

[ok      ] 기대=NO  실제=NO  | 2 더하기 3은 얼마인가?
[ok      ] 기대=NO  실제=NO  | 파이썬에서 리스트와 튜플의 차이는?
[ok      ] 기대=YES 실제=YES | 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
[ok      ] 기대=YES 실제=YES | 11월 24일 김영삼이 대통령 명령으로 제정한 법은?


## Step 6 : RAGAS 평가용 데이터셋 만들기

RAGAS 는 네 가지 자료가 필요합니다.
- `user_input` — 사용자 질문
- `response`   — RAG 가 생성한 답변
- `retrieved_contexts` — RAG 가 참고한 문서들
- `reference`  — 모범 답안 (Ground Truth)

**KorQuAD 는 사람이 작성한 정답이 데이터셋에 이미 포함**되어 있어, `reference` 를 따로 작성할 필요 없이 그대로 가져다 씁니다. 같은 질문 셋을 **Naive RAG** 와 **Advanced RAG** 두 가지로 풀고 결과를 비교합니다.

토큰 비용 통제를 위해 평가 질문은 5개만 사용합니다. (늘리려면 `EVAL_N` 변경)

In [ ]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
EVAL_N = 20  # 평가 질문 수. 표본 분산을 줄이려 20개로 설정. 줄이려면 5~10.
eval_samples = list(raw_ds)[:EVAL_N]
questions = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

# Naive RAG 로 답변 + 컨텍스트 수집
naive_answers, naive_contexts = [], []
for q in questions:
    ctx = naive_retriever.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers.append(a)
    naive_contexts.append([d.page_content for d in ctx])

# Advanced RAG 로 답변 + 컨텍스트 수집
adv_answers, adv_contexts = [], []
for q in questions:
    a, ctx = advanced_rag(q)
    adv_answers.append(a)
    adv_contexts.append([d.page_content for d in ctx])

print(f"데이터셋 준비 완료 — {EVAL_N}개 질문 × 2개 파이프라인")

데이터셋 준비 완료 — 20개 질문 × 2개 파이프라인


In [ ]:
from datasets import Dataset

def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)

## Step 7 : RAGAS로 4대 지표 계산하기  

Judge LLM은 `gpt-4o-mini`로, 임베딩은 `text-embedding-3-small`로 설정합니다.  
(Judge에 더 강한 모델을 쓰면 채점은 더 정교해지지만 비용이 늘어납니다.)

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

judge_llm  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb  = OpenAIEmbeddings(model="text-embedding-3-small")
metrics    = [faithfulness, answer_relevancy,
              context_precision, context_recall]

print("=== Naive RAG 채점 ===")
naive_result = evaluate(naive_ds, metrics=metrics,
                        llm=judge_llm, embeddings=judge_emb,
                        raise_exceptions=False)

print("=== Advanced RAG 채점 ===")
adv_result = evaluate(adv_ds, metrics=metrics,
                      llm=judge_llm, embeddings=judge_emb,
                      raise_exceptions=False)

=== Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

naive_df = naive_result.to_pandas()
adv_df   = adv_result.to_pandas()

def summary(df, label):
    cols = ["faithfulness", "answer_relevancy",
            "context_precision", "context_recall"]
    avg = df[cols].mean()
    avg.name = label
    return avg

compare = pd.concat([summary(naive_df, "Naive RAG"),
                     summary(adv_df,   "Advanced RAG")], axis=1)
print(compare.round(3))
print("\nDelta (Advanced - Naive):")
print((compare["Advanced RAG"] - compare["Naive RAG"]).round(3))

                   Naive RAG  Advanced RAG
faithfulness           0.625         0.900
answer_relevancy       0.302         0.267
context_precision      0.708         0.900
context_recall         0.800         0.900

Delta (Advanced - Naive):
faithfulness         0.275
answer_relevancy    -0.035
context_precision    0.192
context_recall       0.100
dtype: float64


### 결과 해석 가이드

위 비교표를 처음 보면 **‘Advanced 가 더 나쁜 거 아닌가?’** 라는 착각을 하기 쉽습니다. KorQuAD 위에서의 결과 해석 방법을 정리합니다.

**1. `context_precision` 의 개선 (+) 이 Advanced RAG 의 핵심 효과**
검색 결과의 ‘상단’에 정답 문단을 두는 일을 Reranker 가 잘 했다는 의미. Δ가 0.05~0.15 정도면 잘 작동.

**2. `context_recall = 1.0` 으로 포화될 수 있다**
unique context 가 800개 정도면 Naive top-3 에도 정답이 거의 항상 들어옵니다. 이 지표는 더 큰 DB(수만 문서)에서 차이가 드러납니다.

**3. `faithfulness` 가 살짝 떨어질 수 있다**
Reranker 가 컨텍스트를 ‘짧고 집중’ 시키면 LLM이 그 좁은 정보에서 답을 만들 때 일부 주장이 “미뒷받침” 으로 채점되어 점수가 약간 내려갈 수 있음. **정상 범위 (-0.1 이내)**.

**4. `answer_relevancy` 가 0.2~0.4 로 낮은 이유 — KorQuAD 의 구조적 특성**
KorQuAD 정답은 *‘대중교통체계’* 같이 한 단어~한 구절. RAG 답변도 짧게 나오는데, RAGAS 의 `answer_relevancy` 는 **답변에서 질문을 역추론**해 원래 질문과의 유사도를 계산합니다. 답변이 한 단어면 역추론이 흐려져 점수가 낮아집니다. **모델 잘못이 아닌 데이터셋 특성**.

**5. 표본 20개로도 Δ가 ±0.05 이내면 ‘차이 없음’으로 봐야 한다**
20문항에서 ±0.05 는 표본 noise. 더 확실한 판단이 필요하면 `scipy.stats.ttest_rel` 로 통계 검정을 하거나 50~100문항으로 늘려야 합니다.

**6. 한국어 짧은 정답 벤치마크의 한계**
KorQuAD/KLUE-MRC 처럼 정답이 짧은 extractive QA 벤치마크는 `context_precision` 위주로 평가 효과를 봐야 하고, `answer_relevancy` 는 절대값보다 **Naive 대비 상대 변화**로 읽어야 합니다.

### Quiz  
위 표에서 Advanced RAG가 가장 크게 개선한 지표는 무엇인가요? 그리고 그 지표는 우리가 적용한 **어떤 기법**과 가장 직접적으로 연결될까요?  

**Answer (예시)**:  
보통 `context_precision`이 가장 크게 오릅니다. 이는 우리가 추가한 **Reranker**가 ‘진짜 관련도가 높은 문서를 상위에 두는 일’을 잘 했다는 의미입니다.  `context_recall`은 **Multi-Query**가 검색 폭을 넓혔다면 같이 오릅니다.  `faithfulness`와 `answer_relevancy`는 컨텍스트 품질이 올라가면 부수적으로 개선됩니다.

## Step 8 : (선택) 평가 데이터를 LLM으로 자동 생성하기  

현업에서는 모범 답안(`reference`)을 사람이 직접 작성하는 게 가장 큰 부담입니다.  
RAGAS는 **원본 문서만 주면 Question·Reference·Context 한 세트를 자동으로 만들어 주는** 기능을 제공합니다.  
자세한 사용법은 공식 문서를 참고하세요.

https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/

---
# 추가 실습 — KLUE-MRC 한국어 뉴스 MRC 벤치마크로 RAG 평가하기

메인 실습은 위키 기반 **KorQuAD v1** 으로 진행했습니다. 이번 추가 실습은 도메인을 바꿔, **한국어 뉴스 기사 기반의 MRC 벤치마크 KLUE-MRC** 위에서 같은 파이프라인을 처음부터 다시 조립해 봅니다.

**KLUE-MRC**
- 카카오·네이버 등 한국 NLP 팀이 함께 만든 한국어 표준 벤치마크 KLUE 의 MRC 태스크
- 한국어 **뉴스 기사** 기반 (KorQuAD 의 위키와 도메인이 다름)
- 사람이 직접 작성한 정답 포함
- **`is_impossible=True`** 인 답할 수 없는 질문도 일부 포함 → 데이터 필터링이 필요한 도전적 케이스

위키 기반 KorQuAD 와 비교했을 때 어떤 차이(질문 스타일, 검색 난이도, 점수 분포)가 나는지 직접 관찰해 보세요.

이번에도 일부만 샘플링해서 토큰 비용을 통제합니다.
- Vector DB 에 들어갈 unique context: 약 200개
- 평가 질문: 20개
- 예상 비용: GPT-4o-mini 기준 RAGAS 평가까지 합쳐서 약 \$0.10 ~ \$0.20

**📥 데이터셋 다운로드 / 출처**
- HuggingFace `datasets` 자동 다운로드: <https://huggingface.co/datasets/klue>
- KLUE 공식 사이트: <https://klue-benchmark.com/>
- KLUE 논문: <https://arxiv.org/abs/2105.09680>

> 다른 데이터셋으로 한 번 더 해보고 싶다면:  
> - MIRACL 한국어: <https://huggingface.co/datasets/miracl/miracl> (config: `ko`)  
> - 영어 SQuAD: <https://huggingface.co/datasets/rajpurkar/squad>

### Step A. 데이터셋 로드

`datasets` 라이브러리로 KLUE-MRC 를 한 줄에 받아옵니다. KLUE 는 여러 sub-task 가 있는 멀티태스크 벤치마크라서 config 이름 `"mrc"` 를 명시해야 합니다.

데이터셋 페이지: <https://huggingface.co/datasets/klue>

In [ ]:
from datasets import load_dataset

ds_klue = load_dataset("klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
print({k: ds_klue[0][k] for k in ds_klue.column_names})

README.md: 0.00B [00:00, ?B/s]

mrc/train-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

mrc/validation-00000-of-00001.parquet:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17554 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5841 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
{'title': 'BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시', 'context': 'BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 

### Step B. Context 추출 + 중복 제거 (+ is_impossible 필터링)

KLUE-MRC 에는 KorQuAD 에는 없는 **`is_impossible=True`** 케이스가 섞여 있습니다 (= context 만 보고는 답할 수 없는 질문). 평가용 ground_truth 가 비어 있으면 RAGAS 의 `context_recall` 이 깨지므로, 답이 있는 샘플만 남기세요.

- `ds_klue.filter(lambda x: not x["is_impossible"])` 로 답 있는 것만 추리고
- `shuffle(seed=42).select(range(300))` 으로 300개 샘플링
- 그 중 `context` 필드 기준으로 중복 제거 (보통 150~200개)
- 각각을 `Document(page_content=..., metadata={"title": ex["title"]})` 로 감싸 `context_docs` 에 담기

In [ ]:
from langchain_core.documents import Document

# 주의: filter().select() 가 돌려주는 것은 Dataset 이지 Document 리스트가 아니다.
#       그대로 context_docs 로 쓰면 뒤에서 page_content 를 못 찾고 조용히 이상하게 동작한다.
answerable = ds_klue.filter(lambda x: not x["is_impossible"]).shuffle(seed=42)
sampled = answerable.select(range(300))

# context 기준 중복 제거 (같은 기사에 질문이 여러 개 달려 있다)
unique_klue = {}
for ex in sampled:
    if ex["context"] not in unique_klue:
        unique_klue[ex["context"]] = ex["title"]

context_docs = [Document(page_content=c, metadata={"title": t})
                for c, t in unique_klue.items()]

print(f"답 있는 샘플: {len(answerable)} -> 샘플링 300 -> unique context: {len(context_docs)}")
print("첫 문서:", context_docs[0].page_content[:200])

Filter:   0%|          | 0/5841 [00:00<?, ? examples/s]

답 있는 샘플: 4008 -> 샘플링 300 -> unique context: 299
첫 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step C. Embedding + VectorStore

메인 실습에서 만든 `embedding` (`OpenAIEmbeddings(model="text-embedding-3-small")`) 을 그대로 재사용해, `context_docs` 로 새 Chroma DB `db_klue` 를 만드세요. (메인 실습의 `db` 변수를 덮어쓰지 마세요. 비교가 안 됩니다.)

> ⚠️ **batch 적재 필수** — KLUE-MRC 의 뉴스 context 는 평균 토큰 수가 커서, 150개 이상을 한 번에 `Chroma.from_documents` 로 넘기면 OpenAI embeddings 의 **300k 토큰/요청 한도** 에 걸려 `BadRequestError` 가 납니다. 메인 cell 8 처럼 100개씩 batch 로 `add_documents` 호출하세요:
> ```python
> db_klue = Chroma(embedding_function=embedding)
> BATCH = 100
> for i in range(0, len(context_docs), BATCH):
>     db_klue.add_documents(context_docs[i:i+BATCH])
> ```

> 인덱싱 토큰 비용: 약 200개 context × 평균 600 토큰 ≈ **120k 토큰** (≈ \$0.003)

In [ ]:
import tiktoken

# 뉴스 context 는 위키보다 길다. chunk 수가 몇 배로 늘어나는지 먼저 확인해 둔다.
lens = [tiktoken_len(d.page_content) for d in context_docs[:50]]
print(f"context 토큰 수 (앞 50건) 평균 {sum(lens)/len(lens):.0f}, 최대 {max(lens)}")

splitter_klue = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50, length_function=tiktoken_len)
docs_klue = splitter_klue.split_documents(context_docs)
print(f"context {len(context_docs)}개 -> chunk {len(docs_klue)}개 "
      f"(context 당 평균 {len(docs_klue)/len(context_docs):.1f})")

# 메인 실습의 db 를 덮어쓰지 않도록 새 변수에 담는다.
# 뉴스 context 는 토큰이 커서 한 번에 넣으면 요청당 300k 토큰 한도에 걸린다 -> 100개씩 batch
db_klue = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(docs_klue), BATCH):
    db_klue.add_documents(docs_klue[i:i+BATCH])

print("db_klue 인덱싱 완료")

context 토큰 수 (앞 50건) 평균 1077, 최대 2001


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


context 299개 -> chunk 863개 (context 당 평균 2.9)
db_klue 인덱싱 완료


> **여기서 KorQuAD와 차이가 생기는 것 같습니다.**
> 위키 context는 대부분 500토큰 안에 들어가서 chunk 1개가 context 1개였는데, 뉴스 기사는 한 건이 여러 chunk로 쪼개집니다.
> 그러면 정답 문장이 든 chunk는 그중 하나뿐이고, 나머지는 같은 기사에서 온 그럴듯한 오답 후보가 됩니다.
>
> - 검색이 더 어려워질 것 같습니다 (top-3에 정답 chunk가 들어올 확률이 낮아짐 → `context_recall` 하락 요인)
> - 대신 Reranker가 할 일은 많아질 것 같습니다 (같은 기사 chunk들 중에서 진짜 근거를 골라야 하니까)
>
> 실제로 그런지는 Step J 결과에서 확인해보겠습니다.

### Step D. 평가용 질문/정답 세트 추출

Step B 에서 필터링·샘플링한 데이터 중 **앞에서 20개**를 평가용으로 떼어내세요.

- `questions_klue` : 각 샘플의 `question` 필드 (문자열 20개)
- `ground_truths_klue` : 각 샘플의 `answers["text"][0]` (정답이 여러 개일 경우 첫 번째 사용)

> 참고: KLUE-MRC 는 정답이 한 구절~한 문장 단위의 **extractive QA** 입니다. 짧은 정답은 RAGAS 의 `context_recall` 변동성을 키우는 경향이 있으니, 평균을 함께 봐주세요.

In [ ]:
# Step B 에서 샘플링한 300건 중 앞 20건을 평가에 쓴다.
# 같은 300건에서 뽑았으니 정답 기사가 db_klue 에 반드시 들어 있다.
# 평가 질문을 다른 데서 가져오면 정답 문서가 인덱스에 없는 상태로 채점하게 되어,
# 점수가 낮아도 원인이 검색인지 데이터인지 구분할 수 없다.
EVAL_N_KLUE = 20
eval_samples_klue = list(sampled)[:EVAL_N_KLUE]
questions_klue = [ex["question"] for ex in eval_samples_klue]
ground_truths_klue = [ex["answers"]["text"][0] for ex in eval_samples_klue]

assert len(questions_klue) == len(ground_truths_klue) == EVAL_N_KLUE
assert all(g.strip() for g in ground_truths_klue), "빈 정답이 섞였다. is_impossible 필터를 확인할 것"

print("질문 예시:", questions_klue[:3])
print("정답 예시:", ground_truths_klue[:3])

질문 예시: ['국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?', '정유공장 공사는 어느 도시에서 진행되는가?', '가장 먼저 리그 진출 팀이 결정되는 경기의 시작 시간은 언제인가?']
정답 예시: ['두 개', '카르발라', '오후 5시 45분']


### Step E. Naive RAG 베이스라인 (KLUE)

메인 실습의 `RAG_PROMPT` 를 그대로 써도 되고, 뉴스 도메인 특성을 살려 *“기사 본문에 근거해서만 답하세요”* 같은 지시를 추가해도 좋습니다.

- `naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})`
- 체인 구조는 메인 Step 1 과 동일

In [ ]:
# 뉴스 도메인 지시를 한 줄 추가한다. 뉴스는 LLM 이 사전학습으로 이미 아는 사건이 많아서
# "기사 밖 지식을 쓰지 말라" 는 제약이 KorQuAD 때보다 실제로 중요하다.
RAG_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "다음 뉴스 기사 본문을 참고해 질문에 한국어로 간결하게 답하세요. "
    "기사에 없는 내용은 알고 있더라도 쓰지 마세요.\n\n"
    "[기사]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})

def answer_with_klue(docs, question):
    return (RAG_PROMPT_KLUE | llm | StrOutputParser()).invoke(
        {"context": format_docs(docs), "question": question})

# 파이프라인 반환 형태를 (answer, docs) 로 통일해 두면 평가 루프를 하나로 쓸 수 있다.
def naive_rag_klue(question):
    docs = naive_retriever_klue.invoke(question)
    return answer_with_klue(docs, question), docs

TEST_Q_KLUE = questions_klue[0]
ans, ctx = naive_rag_klue(TEST_Q_KLUE)
print("Q:", TEST_Q_KLUE)
print("정답:", ground_truths_klue[0])
print("A:", ans)

Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
정답: 두 개
A: 200여 개의 계좌입니다.


### Step F. Multi-Query Retrieval

메인 Step 2 와 동일하게 `MultiQueryRetriever.from_llm(...)` 으로 KLUE 검색기를 감싸세요. 한국어 질문이 들어가면 gpt-4o-mini 가 한국어로 유사 질문을 만들어 줍니다.

확장 질문 로깅을 켜서 어떤 한국어 변형 질문이 만들어지는지 직접 눈으로 확인하세요.

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever_klue = MultiQueryRetriever.from_llm(
    retriever=db_klue.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

docs_mq_klue = multi_query_retriever_klue.invoke(TEST_Q_KLUE)
print("검색된 문서 수:", len(docs_mq_klue))   # 중복 제거 후라 4~9개 사이
print(docs_mq_klue[0].page_content[:200])

INFO:langchain.retrievers.multi_query:Generated queries: ['1. 국내에서 해킹으로 피해를 입은 리플이 포함된 통장의 수는 몇 개인가요?  ', '2. 한국에서 해킹 사건에 연루된 리플이 있는 통장 수는 얼마인가요?  ', '3. 국내에서 해킹으로 영향을 받은 리플이 있는 계좌의 개수는 어떻게 되나요?']


검색된 문서 수: 5
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


`MultiQueryRetriever`를 써보니 결과를 합집합으로 돌려줘서 순위 정보가 사라졌습니다.
어떤 문서가 여러 쿼리에서 공통으로 상위였는지 알 수가 없어서, Advanced 체인에서는 Step 2.5에서 만든 RRF로 직접 융합하기로 했습니다.
아래 셀에서 KLUE용 쿼리 확장과 HyDE를 한 세트로 묶어두겠습니다.

In [ ]:
# KLUE 용 쿼리 확장 / HyDE 프롬프트 (뉴스 문체로 조정)
SUBQUERY_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "당신은 뉴스 기사 검색을 돕는 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 "
    "4개의 한국어 검색 쿼리를 만드세요. 오직 4개만 한 줄에 하나씩 출력하고 번호나 설명은 붙이지 마세요.\n\n"
    "질문: {question}"
)

fan_out_klue_chain = SUBQUERY_PROMPT_KLUE | llm | StrOutputParser()

def fan_out_queries_klue(question, n=4):
    raw = fan_out_klue_chain.invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]

subs = fan_out_queries_klue(TEST_Q_KLUE)
print("확장 질문:")
for q in subs:
    print(" -", q)

확장 질문:
 - 국내에서 해킹 피해를 입은 리플이 포함된 통장 수는 몇 개인가?
 - 국내에서 해킹으로 영향을 받은 리플 통장의 수는 얼마인가?
 - 해킹을 당한 리플이 있는 국내 통장은 몇 개나 되나?
 - 국내에서 해킹된 리플 통장의 총 개수는 얼마인가?


### Step G. HyDE 직접 구현

메인 Step 3 의 `HYDE_PROMPT` 를 그대로 써도 되고, 뉴스 도메인용으로 *“기자가 쓴 한 문단 형태”* 로 답하라는 지시를 추가해도 됩니다.

`hyde_retrieve_klue(question, k=3)` 함수를 만들고 `db_klue` 위에서 동작하도록 하세요.

In [ ]:
HYDE_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "당신은 기자입니다. 다음 질문에 답하는 내용이 담긴 뉴스 기사 한 문단을 작성하세요. "
    "확실하지 않다면 가장 그럴듯한 형태로 쓰세요.\n\n"
    "질문: {question}\n\n기사 본문:"
)

hyde_generator_klue = HYDE_PROMPT_KLUE | llm | StrOutputParser()

def hyde_retrieve_klue(question, k=3):
    hypothetical = hyde_generator_klue.invoke({"question": question})
    return db_klue.similarity_search(hypothetical, k=k), hypothetical

docs_hyde_klue, hyp_klue = hyde_retrieve_klue(TEST_Q_KLUE)
print("가상 기사:\n", hyp_klue[:300], "\n---")
print("첫 문서:", docs_hyde_klue[0].page_content[:200])

가상 기사:
 최근 국내에서 해킹 사건이 발생하여 리플이 포함된 통장 수가 급증한 것으로 나타났다. 금융감독원에 따르면, 이번 해킹으로 인해 피해를 입은 통장 수는 약 1,200개에 달하며, 이는 리플을 보유한 투자자들에게 큰 충격을 주고 있다. 전문가들은 해킹 사건의 배후에 조직적인 범죄가 있을 가능성을 제기하며, 사용자들에게 보안 강화를 위한 주의를 당부하고 있다. 금융당국은 피해자 지원을 위한 긴급 대책을 마련 중이며, 해킹 경로를 추적하기 위한 수사에 나섰다. 
---
첫 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


> **HyDE를 단독으로 쓰면 위험할 것 같습니다.**
> LLM이 가상 기사에서 엉뚱한 고유명사나 숫자를 지어내면 그 오류가 그대로 검색 쿼리가 되기 때문입니다.
> 위 출력을 보니 실제로 통장 수를 실제와 다르게 지어냈습니다.
> 위키였다면 인물명이나 사건명이 앵커 역할을 해줬을 텐데, 뉴스는 수치 자체가 답인 질문이 많아서 지어낸 숫자가 검색을 끌고 가는 것 같습니다.
> 그래서 다음 Step에서는 HyDE 결과를 원 질문 검색 결과와 RRF로 섞어서 쓰기로 했습니다.

### Step H. Multilingual Cross-encoder Reranker

메인 Step 4 에서 이미 `BAAI/bge-reranker-v2-m3` 같은 다국어 reranker 를 사용하고 있습니다. 추가 실습에서는:

- 메인의 `reranker` 인스턴스를 그대로 재사용하거나
- 다른 다국어 reranker 와 비교해 봐도 좋습니다:
  - `Alibaba-NLP/gte-multilingual-reranker-base` — <https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base>
  - `jinaai/jina-reranker-v2-base-multilingual` — <https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual>

`rerank_klue(query, docs, top_k=3)` 함수를 만드세요. (메인 Step 4 의 `rerank` 와 동일 구조)

In [ ]:
# 메인 Step 4 에서 만든 reranker 인스턴스를 그대로 재사용한다.
# 같은 모델을 다시 로드하면 2GB 를 또 받고 GPU 메모리도 두 배로 쓴다.
def rerank_klue(query, docs, top_k=3):
    if not docs:
        return []
    scores = reranker.predict([(query, d.page_content) for d in docs])
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

cand_klue = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q_KLUE)
top3_klue = rerank_klue(TEST_Q_KLUE, cand_klue, top_k=3)

# 리랭커가 실제로 순서를 바꾸고 있는지 확인. 순위가 그대로면 리랭커를 붙였다는 착각만 남는다.
before = [d.page_content[:40] for d in cand_klue[:3]]
after  = [d.page_content[:40] for d in top3_klue]
print("bi-encoder top3:", before)
print("cross      top3:", after)
print("순서 변경됨:", before != after)

bi-encoder top3: ['국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트', '국내외 근·현대 및 최신 미술 관련 자료를 편리하고 손쉽게 열람할 수 있', '높은 가격으로 리플을 판매하는 다단계 조직이 활동하고 있다. 해당 다단계']
cross      top3: ['국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트', '높은 가격으로 리플을 판매하는 다단계 조직이 활동하고 있다. 해당 다단계', '마이크로소프트(MS)의 웹브라우저 인터넷익스플로러(IE)의 모든 버전(6']
순서 변경됨: True


### Step I. Advanced RAG 체인 (넓게 → Rerank → LLM)

메인 Step 5 흐름과 동일.
1. `db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)` 로 후보 10개
2. `rerank_klue(question, candidates, top_k=3)` 로 좁힘
3. `RAG_PROMPT` + `llm` 으로 답변 생성

함수가 `(answer, top_docs)` 둘 다 반환하도록 만들어 두면 다음 평가 단계에서 그대로 씁니다.

In [ ]:
# 원본 질문 + Multi-Query 변형 + HyDE 가상 기사 = 검색 쿼리 세트
def build_query_set_klue(question, n_query=4, use_hyde=True):
    qs = [question] + fan_out_queries_klue(question, n=n_query)
    if use_hyde:
        qs.append(hyde_generator_klue.invoke({"question": question}))
    return qs


# 중간 단계(리랭커 없음)도 함께 만든다.
# 점수가 움직였을 때 검색 폭이 넓어져서(Fusion) 오른 건지, 순서가 정리돼서(Rerank) 오른 건지 분리해서 보기 위함.
def fusion_rag_klue(question, k_each=5, k_final=3):
    results = [db_klue.similarity_search(q, k=k_each) for q in build_query_set_klue(question)]
    top = reciprocal_rank_fusion(results, k=60, top_k=k_final)
    return answer_with_klue(top, question), top


def advanced_rag_klue(question, k_each=5, k_fusion=10, k_final=3):
    results = [db_klue.similarity_search(q, k=k_each) for q in build_query_set_klue(question)]
    fused = reciprocal_rank_fusion(results, k=60, top_k=k_fusion)
    top = rerank_klue(question, fused, top_k=k_final)
    return answer_with_klue(top, question), top


a_adv, c_adv = advanced_rag_klue(TEST_Q_KLUE)
print("Advanced 답변:", a_adv)
print("컨텍스트 수:", len(c_adv))

Advanced 답변: 200여 개의 계좌입니다.
컨텍스트 수: 3


### Step J. RAGAS 로 Naive vs Advanced 비교

메인 Step 6/7 흐름을 KLUE-MRC 변수(`_klue`) 로 옮겨 동일하게 수행하세요.

1. 20개 질문 각각을 Naive / Advanced 파이프라인에 돌려 답변과 컨텍스트 수집
2. `Dataset.from_dict({...})` 로 `naive_ds_klue`, `adv_ds_klue` 두 개 생성 (키: `user_input / response / retrieved_contexts / reference`)
3. `evaluate(..., metrics=[faithfulness, answer_relevancy, context_precision, context_recall], llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)` 두 번
4. 평균표로 비교

메인의 KorQuAD 결과와 점수가 어떻게 다른지 옆에 같이 적어두면 학습 효과가 큽니다.

In [ ]:
from tqdm.auto import tqdm
from datasets import Dataset

def collect_klue(pipeline_fn, label):
    answers, contexts = [], []
    for q in tqdm(questions_klue, desc=label):
        a, ctx = pipeline_fn(q)
        answers.append(a)
        contexts.append([d.page_content for d in ctx])
    return answers, contexts

runs_klue = {}
runs_klue["Naive"]    = collect_klue(naive_rag_klue,    "Naive")
runs_klue["Fusion"]   = collect_klue(fusion_rag_klue,   "Fusion")
runs_klue["Advanced"] = collect_klue(advanced_rag_klue, "Advanced")

def make_dataset_klue(pair):
    answers, contexts = pair
    return Dataset.from_dict({
        "user_input":         questions_klue,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths_klue,
    })

eval_datasets_klue = {k: make_dataset_klue(v) for k, v in runs_klue.items()}
print({k: len(v) for k, v in eval_datasets_klue.items()})

Naive:   0%|          | 0/20 [00:00<?, ?it/s]

Fusion:   0%|          | 0/20 [00:00<?, ?it/s]

Advanced:   0%|          | 0/20 [00:00<?, ?it/s]

{'Naive': 20, 'Fusion': 20, 'Advanced': 20}


In [ ]:
results_klue, dfs_klue = {}, {}
for name, ds in eval_datasets_klue.items():
    print(f"=== KLUE {name} 채점 ===")
    results_klue[name] = evaluate(ds, metrics=metrics, llm=judge_llm,
                                  embeddings=judge_emb, raise_exceptions=False)
    dfs_klue[name] = results_klue[name].to_pandas()

=== KLUE Naive 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== KLUE Fusion 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== KLUE Advanced 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

In [ ]:
COLS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
compare_klue = pd.DataFrame({n: d[COLS].mean() for n, d in dfs_klue.items()})

print("=== KLUE-MRC (뉴스) ===")
print(compare_klue.round(3))

print("\n--- Naive 대비 델타 ---")
print(compare_klue.drop(columns=["Naive"]).sub(compare_klue["Naive"], axis=0).round(3))

print("\n--- 부품별 기여 분해 ---")
print(pd.DataFrame({
    "Multi-Query+RRF (Naive->Fusion)": compare_klue["Fusion"]   - compare_klue["Naive"],
    "Reranker (Fusion->Advanced)":     compare_klue["Advanced"] - compare_klue["Fusion"],
}).round(3))

=== KLUE-MRC (뉴스) ===
                   Naive  Fusion  Advanced
faithfulness       0.650   0.725     0.650
answer_relevancy   0.202   0.239     0.253
context_precision  0.550   0.525     0.792
context_recall     0.650   0.600     0.800

--- Naive 대비 델타 ---
                   Fusion  Advanced
faithfulness        0.075     0.000
answer_relevancy    0.036     0.051
context_precision  -0.025     0.242
context_recall     -0.050     0.150

--- 부품별 기여 분해 ---
                   Multi-Query+RRF (Naive->Fusion)  \
faithfulness                                 0.075   
answer_relevancy                             0.036   
context_precision                           -0.025   
context_recall                              -0.050   

                   Reranker (Fusion->Advanced)  
faithfulness                            -0.075  
answer_relevancy                         0.014  
context_precision                        0.267  
context_recall                           0.200  


#### KorQuAD와 나란히 놓고 보기

처음에는 메인 실습 결과를 숫자로 옮겨 적었는데, 재실행하면서 Step 7 표와 값이 어긋나는 일이 있었습니다.
그래서 `naive_df` / `adv_df`에서 직접 가져오도록 고쳤습니다.

In [ ]:
# 메인 실습(Step 7)의 KorQuAD 결과를 DataFrame 에서 직접 계산한다.
KORQUAD = pd.DataFrame({
    "Naive":    naive_df[COLS].mean(),
    "Advanced": adv_df[COLS].mean(),
}).reindex(index=COLS)

side = pd.concat([KORQUAD.add_prefix("KorQuAD "),
                  compare_klue[["Naive", "Advanced"]].reindex(index=COLS).add_prefix("KLUE ")],
                 axis=1)
print(side.round(3))

# 인덱스 순서가 어긋나면 조용히 엉뚱한 지표끼리 빠진다. reindex 로 맞춰놓고 뺀다.
kl = compare_klue[["Naive", "Advanced"]].reindex(index=COLS)
print("\n--- Naive->Advanced 개선폭 비교 ---")
print(pd.DataFrame({
    "KorQuAD": KORQUAD["Advanced"] - KORQUAD["Naive"],
    "KLUE":    kl["Advanced"] - kl["Naive"],
}).round(3))

# Step 7 표와 값이 일치하는지 확인. 어긋나면 둘 중 하나가 옛 실행 결과다.
assert abs(KORQUAD.loc["faithfulness", "Naive"] - naive_df["faithfulness"].mean()) < 1e-9
print("\nStep 7 표와 일치 확인 완료")

                   KorQuAD Naive  KorQuAD Advanced  KLUE Naive  KLUE Advanced
faithfulness               0.625             0.900       0.650          0.650
answer_relevancy           0.302             0.267       0.202          0.253
context_precision          0.708             0.900       0.550          0.792
context_recall             0.800             0.900       0.650          0.800

--- Naive->Advanced 개선폭 비교 ---
                   KorQuAD   KLUE
faithfulness         0.275  0.000
answer_relevancy    -0.035  0.051
context_precision    0.192  0.242
context_recall       0.100  0.150

Step 7 표와 일치 확인 완료


### Step K. (선택) 좀 더 큰 샘플로 통계적 신뢰도 확보

질문 20개로는 표본 분산이 커서 Naive vs Advanced 차이가 우연일 수도 있습니다. 토큰 비용이 허용된다면 50~100문항으로 늘려 paired t-test 같은 간단한 통계 검정으로 차이가 유의한지 확인해 보세요.

참고: `scipy.stats.ttest_rel(naive_df["faithfulness"], adv_df["faithfulness"])`

In [ ]:
from scipy import stats

# 20문항 평균은 표본 noise 가 크다. 문항별로 짝지어 검정한다.
print("=== KorQuAD : Naive vs Advanced ===")
for col in COLS:
    a = naive_df[col].astype(float)
    b = adv_df[col].astype(float)
    mask = a.notna() & b.notna()
    if mask.sum() < 3:
        print(f"{col:20s} 유효 표본 부족"); continue
    t, p = stats.ttest_rel(a[mask], b[mask])
    print(f"{col:20s} n={mask.sum():2d}  Δ={b[mask].mean()-a[mask].mean():+.3f}  "
          f"p={p:.3f}  {'유의' if p < 0.05 else '판단 보류'}")

print("\n=== KLUE-MRC : Naive vs Advanced ===")
for col in COLS:
    a = dfs_klue["Naive"][col].astype(float)
    b = dfs_klue["Advanced"][col].astype(float)
    mask = a.notna() & b.notna()
    if mask.sum() < 3:
        print(f"{col:20s} 유효 표본 부족"); continue
    t, p = stats.ttest_rel(a[mask], b[mask])
    print(f"{col:20s} n={mask.sum():2d}  Δ={b[mask].mean()-a[mask].mean():+.3f}  "
          f"p={p:.3f}  {'유의' if p < 0.05 else '판단 보류'}")

=== KorQuAD : Naive vs Advanced ===
faithfulness         n=20  Δ=+0.275  p=0.012  유의
answer_relevancy     n=20  Δ=-0.035  p=0.200  판단 보류
context_precision    n=20  Δ=+0.192  p=0.022  유의
context_recall       n=20  Δ=+0.100  p=0.163  판단 보류

=== KLUE-MRC : Naive vs Advanced ===
faithfulness         n=20  Δ=+0.000  p=1.000  판단 보류
answer_relevancy     n=20  Δ=+0.051  p=0.073  판단 보류
context_precision    n=20  Δ=+0.242  p=0.012  유의
context_recall       n=20  Δ=+0.150  p=0.083  판단 보류


### Step L. (추가 실험) `is_impossible` — 답할 수 없는 질문

답이 없는 질문은 `reference`가 빈 문자열이라 RAGAS 4대 지표를 그대로 쓸 수가 없었습니다.
그래서 지표를 바꿔서, 모델이 "기사에 없다"고 말하는 비율(abstention rate)을 재보기로 했습니다.

생각해보면 실무에서는 이쪽이 더 중요할 것 같습니다.
답할 수 없는 질문에 검색된 다른 기사 내용으로 그럴듯한 답을 조립하면, 답변 자체는 검색 문서에 근거하고 있으니
`faithfulness`는 높게 나올 텐데, 사용자가 받는 답은 틀린 답이기 때문입니다.
실제로 그런 경우가 나오는지 확인해보겠습니다.

In [ ]:
impossible = ds_klue.filter(lambda x: x["is_impossible"]).shuffle(seed=42).select(range(10))

ABSTAIN_HINTS = ["없", "확인할 수 없", "알 수 없", "언급되지", "나와 있지", "제공되지"]

abstain = 0
for ex in impossible:
    ans, _ = naive_rag_klue(ex["question"])
    said_no = any(h in ans for h in ABSTAIN_HINTS)
    abstain += said_no
    print(f"[{'거부' if said_no else '답변생성'}] {ex['question'][:35]} -> {ans[:60]}")

print(f"\n답 없는 질문 10건 중 거부 {abstain}건 (거부율 {abstain/10:.0%})")

Filter:   0%|          | 0/5841 [00:00<?, ? examples/s]

[답변생성] 웹 드라마가 끝난 달은? -> 웹 드라마 '간서치열전'은 최근 종방하였고, '연애세포'는 지난 2일 첫 공개되었습니다. 따라서 웹 드라마가
[답변생성] 매해 연구개발 인력을 축소시키고 있다고 한 사람은? -> 이인재 한국노동연구원장입니다.
[거부] 가계 재무건전성이 매우 위험하다고 분석한 기관은? -> 기사에 해당 내용이 없습니다.
[거부] 카페베네 첫 창립주년을 기념하여 홈페이지를 개편한 날짜는? -> 기사에 해당 내용이 없습니다.
[거부] 수처리 사업을 선진국 위주로 넓혀갈 예정이라고 발언한 사람은? -> 기사에 해당 내용이 없습니다.
[거부] 고덕주공 4단지를 재건축한 4000만원대의 아파트는? -> 기사에 고덕주공 4단지에 대한 내용이 없으므로 답변할 수 없습니다.
[거부] 조조가 동관으로 진격을 시작한 시기는? -> 기사에 조조가 동관으로 진격을 시작한 시기에 대한 내용이 없습니다.
[거부] 퇴직소득세에 대한 누진과세 기준이 종전 연봉 700만원 수준에서 -> 기사에 해당 내용이 없으므로 답변할 수 없습니다.
[거부] 수비크조선소가 필리핀에 세금을 납부한 기간은? -> 기사에 해당 내용이 없으므로 답변할 수 없습니다.
[거부] 국가 기록물법 개정안으로 검찰이 수사 근거를 얻기 쉬워질것이라  -> 기사에는 국가 기록물법 개정안으로 검찰이 수사 근거를 얻기 쉬워질 것이라고 말한 사람에 대한 정보가 없습니다

답 없는 질문 10건 중 거부 8건 (거부율 80%)


### 마지막 Quiz — 답변

이번 실행 결과를 먼저 정리해봤습니다.

| 지표 | KorQuAD Naive | KorQuAD Adv | KLUE Naive | KLUE Fusion | KLUE Adv |
|---|---|---|---|---|---|
| faithfulness | 0.625 | 0.900 | 0.650 | 0.725 | 0.650 |
| answer_relevancy | 0.302 | 0.267 | 0.202 | 0.239 | 0.253 |
| context_precision | 0.708 | 0.900 | 0.550 | 0.525 | 0.792 |
| context_recall | 0.800 | 0.900 | 0.650 | 0.600 | 0.800 |

**1. 도메인 비교 — 가장 크게 달라진 지표**

`context_precision`이 가장 크게 달라졌습니다. KLUE Naive가 0.550으로, KorQuAD Naive(0.708)보다 뚜렷하게 낮았습니다.
같은 검색 전략을 썼는데 뉴스에서는 정답 문서를 상위에 올리는 게 더 어려웠던 것 같습니다.

왜 그런지 생각해보니 뉴스의 특성 세 가지가 걸렸습니다.

- **기사 하나가 여러 chunk로 쪼개집니다.** Step C 출력을 보니 KorQuAD는 context 847개 → chunk 1264개(약 1.5배)인데
  KLUE는 299개 → 863개로 2.9배였습니다. 그러면 정답 문장을 가진 chunk는 하나뿐이고 나머지는 같은 기사에서 온,
  어휘가 거의 똑같은 오답이 됩니다. bi-encoder는 이 셋을 구분할 근거가 벡터 안에 없을 것 같습니다.
- **수치나 날짜가 정답인 질문이 많습니다.** Step G에서 HyDE가 만든 가상 기사는 "피해를 입은 통장 수는 약 1,200개에 달하며"라고 썼는데
  실제 정답은 그 값이 아니었습니다. 위키였다면 인물명이 앵커가 되어 HyDE가 틀려도 근처로 갔을 텐데 뉴스는 그렇지 않은 것 같습니다.
- **확장 질문의 다양성이 낮았습니다.** Step F 출력을 보니 변형 4개가 사실상 같은 말이었습니다.
  이러면 RRF가 같은 결과를 여러 번 세면서 순위만 굳히게 될 것 같습니다.

**2. Advanced 효과 — KorQuAD와 같았나**

개선폭 자체는 KLUE가 더 컸습니다(`context_precision` KorQuAD +0.192 vs KLUE +0.242).
Naive가 낮은 지점에서 출발했으니 회복할 여지가 컸던 것 같습니다.

그런데 단계별로 나눠 보니 두 도메인의 이야기가 달랐습니다(Step J 분해표).
이렇게 구성요소를 하나씩 빼거나 더하면서 기여를 재는 걸 ablation이라고 부른다는 것도 이번에 알게 됐습니다.

| 지표 | Multi-Query+RRF 단계<br>(Naive→Fusion) | Reranker 단계<br>(Fusion→Advanced) |
|---|---|---|
| faithfulness | +0.075 | −0.075 |
| context_precision | −0.025 | +0.267 |
| context_recall | −0.050 | +0.200 |

예상 밖이었던 건 KLUE에서 **Multi-Query+RRF가 검색 지표를 오히려 떨어뜨렸다**는 점입니다(precision −0.025, recall −0.050).
쿼리를 5개 더 만들고 LLM을 다섯 번 더 호출했는데 손해를 본 셈입니다.
1번에서 본 원인(지어낸 숫자, 같은 말 반복하는 변형)과 연결되는 것 같습니다.
개선은 전부 리랭커가 만들었습니다(precision +0.267, recall +0.200).

이걸 보고 나서, "Advanced RAG를 적용했다"는 말만으로는 아무것도 설명이 안 된다는 걸 알게 됐습니다.
어느 단계가 이 도메인에서 실제로 기여하는지가 중요하고, 그걸 알려면 중간 단계(Fusion)를 따로 측정해야 했습니다.
KorQuAD 쪽은 Naive/Advanced 두 줄만 뽑아서 같은 분해를 못 했는데, 이건 다음 과제로 남겨두려고 합니다.

**`faithfulness`가 두 도메인에서 반대로 움직인 것도 예상 밖이었습니다.**
KorQuAD에서는 +0.275(p=0.012)로 크게 올랐습니다. 리랭커로 컨텍스트가 짧아지면 근거로 삼을 문장이 줄어드니
오히려 내려갈 거라고 생각했는데 반대였습니다.
찾아보니 검색이 엉뚱한 문서를 물어오면 LLM이 그 안에서 답을 조립하게 되고, 그게 미뒷받침으로 채점되기 때문인 것 같습니다.
반면 KLUE에서는 리랭커 단계가 −0.075로 제가 원래 예상했던 방향으로 움직였고, Fusion의 +0.075와 상쇄되어 순 변화가 0.000(p=1.000)이 됐습니다.

정리하면 `faithfulness`는 이름만 보면 생성 단계 지표 같은데 실제로는 검색 품질에 많이 의존하는 것 같습니다.
검색이 나쁠 때는 검색이 좋아지면 같이 오르고, 검색이 이미 괜찮을 때는 컨텍스트가 줄어드는 효과가 드러나는 것으로 보입니다.
지표 이름만 보고 어느 구간 담당인지 정하면 원인을 잘못 짚을 수 있겠다는 생각이 들었습니다.

한편 `context_recall`은 두 도메인 다 올랐지만(+0.100, +0.150) t-test에서는 유의하지 않았습니다(p=0.163, p=0.083).
20문항에서 +0.100이면 문항 두 개 차이라, 평균만 보고 개선됐다고 쓰면 안 되는 것 같습니다.

**3. `is_impossible` 케이스 — 어떤 지표가 망가지나**

정답이 빈 문자열이라 `context_recall`은 계산 자체가 안 됩니다(참조 답변에서 주장을 뽑을 수가 없으니까요).
`context_precision`도 기준이 없어집니다.

그런데 더 중요한 건 `faithfulness`는 망가지지 않는다는 점이었습니다.
모델이 검색된 다른 기사 내용으로 답을 조립하면 답변은 문서에 근거하고 있으니 점수가 높게 나옵니다.
Step L에서 거부율은 80%(10건 중 8건)였는데, 거부하지 않은 2건이 정확히 이런 경우였습니다.

```
[답변생성] 웹 드라마가 끝난 달은?
        -> 웹 드라마 '간서치열전'은 최근 종방하였고, '연애세포'는 지난 2일 첫 공개되었습니다. 따라서 ...

[답변생성] 매해 연구개발 인력을 축소시키고 있다고 한 사람은?
        -> 이인재 한국노동연구원장입니다.
```

둘 다 답할 수 없는 질문인데 검색된 기사에서 조각을 뽑아 단정적으로 답했습니다.
4대 지표를 다 통과하면서도 사용자에게는 틀린 답이 나가는 경로가 있다는 걸 확인했고,
답할 수 없는 질문이 섞인 데이터라면 거부율 같은 다른 축의 지표를 같이 봐야겠다고 생각했습니다.

**4. (선택) MIRACL ko로 옮기면**

MIRACL은 위키 기반이지만 KorQuAD와 성격이 다르다고 합니다. extractive QA가 아니라 문서 검색(retrieval) 벤치마크라
정답이 정답 문단 집합으로 주어지고, 코퍼스 규모도 수십만 문서 단위라고 합니다. 이번 결과를 바탕으로 예상해보면 이렇습니다.

- `context_recall`이 많이 떨어질 것 같습니다. 이번 실습은 DB가 800~900 chunk라 Naive top-3에도 정답이 자주 들어왔는데,
  코퍼스가 수십만이면 검색 폭을 넓히는 것 자체의 가치가 커질 것 같습니다.
  그러면 **KLUE에서 손해였던 Multi-Query+RRF가 여기서는 이득이 될 수도 있겠다**는 생각이 들었습니다.
- 반대로 Reranker 효과는 줄어들 것 같습니다. 후보 안에 정답이 없으면 아무리 잘 정렬해도 소용없을 테니까요.
- `answer_relevancy`는 정답이 문단 단위라 이번보다 절대값이 높게 나올 것 같습니다.
  짧은 정답이 역추론을 흐리는 문제가 덜할 것 같습니다.

## 마치며

이번 실습에서는 한국어 QA 벤치마크 위에서 다음을 진행했습니다.

- **KorQuAD v1** 위에 Naive RAG 베이스라인 구성
- Multi-Query / **RAG-Fusion (RRF)** / HyDE / Cross-encoder Reranking 적용
- ‘넓게 검색 → Reranker 로 좁힘 → LLM 답변’ Advanced RAG 체인 조립
- **Self-RAG** 패턴 — 검색 필요성 판단 + 답변 자가 비평 + HyDE 재시도
- RAGAS 4대 지표로 Naive vs Advanced 를 정량 비교
- 추가 실습으로 도메인을 옮긴 **KLUE-MRC (뉴스 기반 한국어 MRC)** 에서 같은 파이프라인 재구성

**다음 Day 3 에서는** RAG 가 LLM Agent 와 결합되어 ‘검색 자체를 계획하고 도구를 쓰는’ Agentic RAG 로 진화하는 흐름을 다룹니다.


## 회고

**측정 결과**

KorQuAD에서 Advanced RAG는 `faithfulness` +0.275(p=0.012), `context_precision` +0.192(p=0.022)로 유의한 개선이 나왔습니다.
KLUE에서는 `context_precision` +0.242(p=0.012)만 유의했습니다.
`context_recall`은 두 도메인 다 올랐는데 p값이 0.163, 0.083이라 개선됐다고 말하기는 어려웠습니다.
평균표만 봤으면 그냥 올랐다고 적었을 텐데, 20문항에서 +0.100이면 문항 두 개 차이라는 걸 t-test를 붙이고서야 알았습니다.

`answer_relevancy`는 KorQuAD에서 −0.035로 오히려 내려갔습니다.
성능 문제라기보다는 정답이 "대중교통체계" 같은 한 단어라, 답변에서 질문을 역추론하는 이 지표의 계산 방식이
잘 안 맞는 것 같습니다. 이 벤치마크에서는 이 지표로 뭘 판단하기 어렵겠다는 생각이 들었습니다.